<a href="https://colab.research.google.com/github/encoras/Introduction-to-OpenCV/blob/master/amber_yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
import os
import shutil
import random
import string
from pathlib import Path
from tqdm import tqdm   # gražus progresas

# 1. Prijungiam Google Drive

# 2. Keliai
SOURCE_DIR = "/content/drive/MyDrive/Colab Notebooks/kevir_12_mazi"
TARGET_DIR = "/content/orig_amber_dataset"

# 3. Sukuriame tikslinę direktoriją (jei nėra)
os.makedirs(TARGET_DIR, exist_ok=True)

# 4. Funkcija generuoti trumpą random priesagą
def random_suffix(length=6):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))

# 5. Kopijuojame visus vaizdus su unikaliais vardais
print("Kopijuojame failus ir pridedame unikalius priesagas...\n")

copied_count = 0
skipped_count = 0

# einame per visus failus rekursyviai
for root, dirs, files in os.walk(SOURCE_DIR):
    for file in tqdm(files, desc="Apdorojama"):
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            old_path = os.path.join(root, file)

            # išsaugome santykinį kelią, kad išlaikytume struktūrą nk01/nk02/...
            rel_path = os.path.relpath(root, SOURCE_DIR)
            target_subdir = os.path.join(TARGET_DIR, rel_path)
            os.makedirs(target_subdir, exist_ok=True)

            # naujas vardas: originalus_stem + _rnd_XXXXXX + plėtinys
            base, ext = os.path.splitext(file)
            new_filename = f"{base}_rnd_{random_suffix()}{ext}"
            new_path = os.path.join(target_subdir, new_filename)

            # kopijuojame
            try:
                shutil.copy2(old_path, new_path)
                copied_count += 1
                # print(f"OK: {file} → {new_filename}")
            except Exception as e:
                print(f"Klaida kopijuojant {file}: {e}")
                skipped_count += 1

print("\n" + "="*60)
print(f"Baigta. Nukopijuota failų: {copied_count}")
print(f"Praleista / klaidų: {skipped_count}")
print(f"Failai dabar yra: {TARGET_DIR}")
print("="*60)

# Papildomai – patikriname kiek failų kiekvienoje poaplankėje
print("\nFailų kiekis pagal aplankus:")
for subdir in sorted(os.listdir(TARGET_DIR)):
    sub_path = os.path.join(TARGET_DIR, subdir)
    if os.path.isdir(sub_path):
        count = len([f for f in os.listdir(sub_path) if f.lower().endswith(('.png','.jpg','.jpeg'))])
        print(f"{subdir:8} : {count:4} failų")

Kopijuojame failus ir pridedame unikalius priesagas...



Apdorojama: 100%|██████████| 199/199 [03:13<00:00,  1.03it/s]


Baigta. Nukopijuota failų: 2343
Praleista / klaidų: 0
Failai dabar yra: /content/orig_amber_dataset

Failų kiekis pagal aplankus:
NK01     :  185 failų
NK02     :  190 failų
NK03     :  199 failų
NK04     :  196 failų
NK05     :  199 failų
NK06     :  195 failų
NK07     :  197 failų
NK08     :  195 failų
NK09     :  196 failų
NK10     :  198 failų
NK11     :  195 failų
NK12     :  198 failų


In [ ]:
import cv2
import numpy as np
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split
import random

# ========================== KONFIGŪRACIJA ==========================
DATASET_PATH = "/content/orig_amber_dataset"          # ← tavo nk01-nk12 direktorija
YOLO_DATASET = "/content/yolo_amber_dataset"    # naujas YOLO formatas

# HSV ribos (plačios, kad pagautų visus gintarus)
LOWER_AMBER = np.array([5,  30,  20])
UPPER_AMBER = np.array([40, 255, 255])

KERNEL = np.ones((5,5), np.uint8)

# ========================== SEGMENTAVIMO FUNKCIJA (atnaujinta) ==========================
def segment_amber(image_path):
    img = cv2.imread(str(image_path))
    if img is None:
        return None, None

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOWER_AMBER, UPPER_AMBER)

    # Stipresnis valymas tamsiems gintarams
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, KERNEL, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  KERNEL, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img, None

    # Imame tik didžiausią kontūrą
    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < 100:  # filtras nuo triukšmo
        return img, None

    mask_clean = np.zeros(mask.shape, np.uint8)
    cv2.drawContours(mask_clean, [largest], -1, 255, cv2.FILLED)

    return img, mask_clean

# ========================== YOLO BBOX GENERAVIMAS ==========================
def get_yolo_bbox(img, mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)

    height, width = img.shape[:2]

    x_center = (x + w / 2) / width
    y_center = (y + h / 2) / height
    norm_w   = w / width
    norm_h   = h / height

    return f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"   # kol kas klasė 0 (vėliau pakeisime)

# ========================== KURIAME YOLO STRUKTŪRĄ + ANOTACIJAS ==========================
print("🚀 Generuojame YOLO anotacijas...")

# Sukuriame tuščias direktorijas
for split in ['train', 'val']:
    (Path(YOLO_DATASET) / "images" / split).mkdir(parents=True, exist_ok=True)
    (Path(YOLO_DATASET) / "labels" / split).mkdir(parents=True, exist_ok=True)

all_images = []
all_class_ids = []

for nk_folder in sorted(Path(DATASET_PATH).glob("NK*")):
    class_name = nk_folder.name
    class_id = int(class_name.replace("NK", "")) - 1   # nk01 → 0, nk02 → 1, ..., nk12 → 11

    for img_path in nk_folder.glob("*.*"):
        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        img, mask = segment_amber(img_path)
        if img is None or mask is None:
            continue

        yolo_line = get_yolo_bbox(img, mask)


        if not yolo_line:
            continue

        # Pridedame klasės ID (vietoj "0")
        yolo_line = str(class_id) + yolo_line[1:]

        all_images.append(img_path)
        all_class_ids.append(class_id)

# 80/20 train/val split (stratified pagal klases)
train_idx, val_idx = train_test_split(range(len(all_images)), test_size=0.20,
                                      stratify=all_class_ids, random_state=42)

print(f"✅ Rasta {len(all_images)} tinkamų gintarų. Train: {len(train_idx)}, Val: {len(val_idx)}")

# Kopijuojame failus + rašome .txt
for idx, img_path in enumerate(all_images):
    class_id = all_class_ids[idx]
    yolo_line = get_yolo_bbox(cv2.imread(str(img_path)), segment_amber(img_path)[1])
    yolo_line = str(class_id) + yolo_line[1:]

    is_train = idx in train_idx
    split = "train" if is_train else "val"

    # Kopijuojame nuotrauką
    new_img_path = Path(YOLO_DATASET) / "images" / split / img_path.name
    shutil.copy(img_path, new_img_path)

    # Rašome YOLO .txt failą
    txt_path = Path(YOLO_DATASET) / "labels" / split / (img_path.stem + ".txt")
    with open(txt_path, "w") as f:
        f.write(yolo_line + "\n")

print(f"✅ YOLO duomenų rinkinys paruoštas: {YOLO_DATASET}")

🚀 Generuojame YOLO anotacijas...
✅ Rasta 2270 tinkamų gintarų. Train: 1816, Val: 454


In [ ]:
!ls -l /content/yolo_amber_dataset/images/train | wc -l
!ls -l /content/yolo_amber_dataset/labels/train | wc -l
!ls -l /content/yolo_amber_dataset/images/val | wc -l
!ls -l /content/yolo_amber_dataset/labels/val | wc -l

In [5]:
data_yaml = """
path: /content/yolo_amber_dataset
train: images/train
val: images/val

nc: 12
names: ['nk01','nk02','nk03','nk04','nk05','nk06','nk07','nk08','nk09','nk10','nk11','nk12']
"""

with open("/content/data.yaml", "w") as f:
    f.write(data_yaml)

In [6]:
# 1. Įdiegimas
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.6 MB/s eta 0:00:00


In [7]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")        # arba yolov8s.pt
model.train(data="/content/data.yaml", epochs=50, imgsz=640, batch=16, patience=30)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, i

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7920c248d130>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,  